In [1]:
import polars as pl
from collections import defaultdict
from tqdm.notebook import tqdm

In [2]:
articles_path='../data/articles.parquet'
customer_path='../data/customers.parquet'
transaction_path='../data/transactions.parquet'

In [3]:
articles=pl.read_parquet(articles_path)
transactions=pl.read_parquet(transaction_path)

In [4]:
def recall_repurchase_decay(data,topk=50,hist_len=100):
    DAY=86400

    dfs=[]

    for cid ,g in tqdm(data.group_by('customer_id'),total=data['customer_id'].unique().shape[0]):
        g=g.sort('time',descending=True).head(hist_len)
        cid=str(cid[0])
        scores=defaultdict(float)
        max_time=g['time'].max()

        for aid,t in zip(g['article_id'],g['time']):
            dt=(max_time-t)/DAY
            scores[aid]+=1/(1+dt)

        res=sorted(scores.items(),key=lambda x:x[1],reverse=True)[:topk]

        dfs.append(
            pl.DataFrame({
                "customer_id": [cid] * len(res),
                "article_id": [aid for aid, _ in res],
                "rank": list(range(len(res)))
            })
        )

    return pl.concat(dfs).with_columns(
        pl.col("rank").cast(pl.UInt8)
    )

In [5]:
res=recall_repurchase_decay(transactions)

  0%|          | 0/1362281 [00:00<?, ?it/s]

In [6]:
res.to_pandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21233612 entries, 0 to 21233611
Data columns (total 3 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   customer_id  object
 1   article_id   int64 
 2   rank         uint8 
dtypes: int64(1), object(1), uint8(1)
memory usage: 344.2+ MB


In [7]:
def get_validation_data(data: pl.DataFrame):
    DAY = 86400
    WEEK = 7 * DAY

    max_time = data.select(pl.col("time").max()).item()

    valid_start = max_time - 6 * DAY
    train_start = valid_start - 6 * WEEK

    train_df = data.filter(
        (pl.col("time") >= train_start) &
        (pl.col("time") <  valid_start)
    )

    valid_df = data.filter(
        pl.col("time") >= valid_start
    )

    return train_df, valid_df

In [8]:
def metric_recall(data,topk=5):
    train_df,valid_df=get_validation_data(data)

    user_item=recall_repurchase_decay(train_df,topk*10)

    pred_df = (
        user_item
        .sort("rank")
        .group_by("customer_id")
        .agg(pl.col("article_id").alias("pred_items"))
    )

    true_df = (
        valid_df
    .group_by("customer_id")
        .agg(pl.col('article_id').unique().alias("true_items")))

    eval_df=pred_df.join(true_df,on='customer_id',how='inner')

    for k in range(10, topk * 10 + 1, 10):
        total = 0.0
        cnt = 0

        for pred, true in zip(eval_df["pred_items"], eval_df["true_items"]):
            if len(true) == 0:
                continue
            total += len(set(pred[:k]) & set(true)) / len(true)
            cnt += 1

        print(f"Recall@{k}: {total / cnt:.6f}")

In [9]:
metric_recall(transactions)

  0%|          | 0/312215 [00:00<?, ?it/s]

Recall@10: 0.061001
Recall@20: 0.063166
Recall@30: 0.063403
Recall@40: 0.063441
Recall@50: 0.063458


In [10]:
res.write_parquet('../save/candidate/recall_repurchase.parquet')